# 27 (MLA) — Tune, Select & Score

**ML Analyst perspective.** Search hyperparameters with `CrossValidator` and `TrainValidationSplit`, pick the best model, then **batch-score new applicants** via SQL pushdown and persist the scored table.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Load splits + build a grid

`ParamGridBuilder` produces the Cartesian product of candidate values.

In [ ]:
train = session.table("mla_train")
test = session.table("mla_test")
from irispark.ml.classification import LogisticRegression
from irispark.ml.tuning import ParamGridBuilder

feats = ["renda_std", "idade_std", "historico_idx_std", "divida_ratio_std"]
lr = LogisticRegression(featuresCol=feats, labelCol="inadimplente", maxIter=500, learningRate=0.5)
grid = ParamGridBuilder()
grid.addGrid(lr.getParam("regParam"), [0.0, 0.1, 1.0])
grid.addGrid(lr.getParam("learningRate"), [0.3, 0.5])
print("grid size:", len(grid.build()))

## 2. Cross-validate

3-fold CV averages the metric per param map, then retrains the best on the full train set.

In [ ]:
from irispark.ml.evaluation import BinaryClassificationEvaluator
from irispark.ml.tuning import CrossValidator

eval_ = BinaryClassificationEvaluator(predictionCol="prediction", labelCol="inadimplente", metricName="areaUnderROC")
cv = CrossValidator(estimator=lr, estimatorParamMaps=grid.build(), evaluator=eval_, numFolds=3)
best_cv = cv.fit(train)
print("best params (CV):", cv.bestParams)
print("avg metrics:", [round(m, 3) for m in cv.avgMetrics])

## 3. Train/validation split

A single split is faster but noisier than CV.

In [ ]:
from irispark.ml.tuning import TrainValidationSplit

tvs = TrainValidationSplit(estimator=lr, estimatorParamMaps=grid.build(), evaluator=eval_, trainRatio=0.75)
best_tvs = tvs.fit(train)
print("best params (TVS):", tvs.bestParams)

## 4. Score the held-out test set

Evaluate the CV-selected model on data it never saw.

In [ ]:
test_pred = best_cv.transform(test)
acc = BinaryClassificationEvaluator(predictionCol="prediction", labelCol="inadimplente", metricName="accuracy").evaluate(test_pred)
auc = BinaryClassificationEvaluator(predictionCol="probability", labelCol="inadimplente", metricName="areaUnderROC").evaluate(test_pred)
print(f"test accuracy={acc:.3f}  auc={auc:.3f}")

## 5. Batch-score new applicants

Score a fresh applicants table via SQL pushdown — no data leaves IRIS.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(99)
new = session.createDataFrame(pd.DataFrame({
    "cliente_id": range(1001, 1011),
    "renda_std": rng.normal(0, 1, 10),
    "idade_std": rng.normal(0, 1, 10),
    "historico_idx_std": rng.normal(0, 1, 10),
    "divida_ratio_std": rng.normal(0, 1, 10),
}))
scored = best_cv.transform(new)
scored.select("cliente_id", "probability", "prediction").show()

## 6. Persist the scored table

Save the scored applicants for downstream use, then reload to confirm.

In [ ]:
scored.write.mode("overwrite").saveAsTable("mla_scored")
reloaded = session.table("mla_scored")
print("scored rows:", reloaded.count())
reloaded.select("cliente_id", "prediction").show(5)

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")